# Homework: Agentic RAG

## 0. Autoreload

In [10]:
%load_ext autoreload
%autoreload 2

## 1. Preparation

In [2]:


from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print ("type(files[0]):", type(files[0]))
print ("type(documents[0]) :", type(documents[0]))
print ("documents[0]:", documents[0])
print("document[0].keys():", documents[0].keys())
print ("documents[0]['content']:", documents[0]["content"])

type(files[0]): <class 'gitsource.github.RawRepositoryFile'>
type(documents[0]) : <class 'dict'>
documents[0]: {'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsA

In [ ]:
print("Number of documents:", len(documents))

## 2.Indexing

In [4]:
from sqlitesearch import TextSearchIndex
from gitsource import chunk_documents
import time    

index = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lessons.db"
)

# to count total number of chunks across all documents
total_chunks_count = 0

for doc in documents:
    filename = doc["filename"]

    # chunk document and add individual chunks to the index
    text_chunks = chunk_documents([doc], size=2000, step=1000)
    for i, chunk in enumerate(text_chunks):
        total_chunks_count = total_chunks_count + 1
        chunc_doc = {
            "filename" : filename,
            "content" : chunk["content"]}
        index.add(chunc_doc)
        print(f"Added chunk {i+1}/{len(text_chunks)} for: {filename}")
        time.sleep(0.5)

index.close()
print(f"Done. Chunked Index with {total_chunks_count} chunks saved to lessons.db")



Added chunk 1/3 for: 01-agentic-rag/lessons/01-intro.md
Added chunk 2/3 for: 01-agentic-rag/lessons/01-intro.md
Added chunk 3/3 for: 01-agentic-rag/lessons/01-intro.md
Added chunk 1/3 for: 01-agentic-rag/lessons/02-environment.md
Added chunk 2/3 for: 01-agentic-rag/lessons/02-environment.md
Added chunk 3/3 for: 01-agentic-rag/lessons/02-environment.md
Added chunk 1/5 for: 01-agentic-rag/lessons/03-rag.md
Added chunk 2/5 for: 01-agentic-rag/lessons/03-rag.md
Added chunk 3/5 for: 01-agentic-rag/lessons/03-rag.md
Added chunk 4/5 for: 01-agentic-rag/lessons/03-rag.md
Added chunk 5/5 for: 01-agentic-rag/lessons/03-rag.md
Added chunk 1/3 for: 01-agentic-rag/lessons/04-dataset.md
Added chunk 2/3 for: 01-agentic-rag/lessons/04-dataset.md
Added chunk 3/3 for: 01-agentic-rag/lessons/04-dataset.md
Added chunk 1/7 for: 01-agentic-rag/lessons/05-search.md
Added chunk 2/7 for: 01-agentic-rag/lessons/05-search.md
Added chunk 3/7 for: 01-agentic-rag/lessons/05-search.md
Added chunk 4/7 for: 01-agentic

## 3. Searching

In [1]:
from sqlitesearch import TextSearchIndex

# connect to persistent sqlite db
sqlite_index = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lessons.db"
)

sqlite_index.count()

results = sqlite_index.search("How does the agentic loop keep calling the model until it stops?", num_results=5)
[doc["filename"] for doc in results][0]

'01-agentic-rag/lessons/14-agentic-loop.md'

## 4. Naive RAG

In [2]:
from dotenv import load_dotenv
from rag_helper import RAGBase
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    base_url="http://192.168.8.102:11434/v1",
    api_key="ollama"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    model="llama3.1:8b-instruct-q4_K_M"
)

answer = assistant.naive_rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

--- Token Usage Metric ---
Prompt Tokens:     2308
Completion Tokens: 383
Total Tokens:      2691
--------------------------
The agentic loop keeps calling the model until it stops by using a while loop that continues indefinitely (True) as long as there are function calls in the response from the model. The exit condition is when there are no more function calls, indicating that the model has returned a final answer with no more tool calls.

Here's a simplified version of the code:

```python
it = 1

while True:
    print(f"iteration #{it}...")
    
    # call the API and get the response
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )
    
    # check if there are any function calls in the response
    has_function_calls = False
    
    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make

## 5. Agent RAG

In [3]:
from rag_helper import RAGBase
# from IPython.display import display, Markdown
import os
import aisuite as ai

# aisuite setup
os.environ["OLLAMA_API_URL"] = "http://192.168.8.102:11434"
model = "ollama:llama3.1:8b-instruct-q4_K_M"
llm_client = ai.Client()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=llm_client,
    model=model
)

answer = assistant.agent_rag("How does the agentic loop keep calling the model until it stops?", max_turns=10)
print ("answer:", answer)
# display(Markdown(answer))

[INFO] Token usage structure unavailable for this specific provider.
Number of Searches: 1
answer: The agentic loop keeps calling the model until it stops by using a loop that continues to call the model and run tools until it's done. An agent has three parts: instructions (the role and behavior of the agent), tools (the functions the agent can call to carry out the task, which in this case is `search`), and memory (the message history).

Here's how it works:

1. The model decides to invoke `search`, which is one of the available tools.
2. The agent sends a prompt to search for relevant information, along with its own memory (which includes previous messages exchanged between the user and the model).
3. The model executes the function call, which in this case is `search`.
4. The result of the search is then sent back to the agent as part of the conversation history.
5. Based on the results received from the LLM, the agentic loop will continue calling tools (such as search) until it run